# Exercise 1: Data Profiling

**Objective:** Explore the dirty dataset and identify quality issues using SQL before applying Great Expectations.

In this exercise, you will connect directly to the PostgreSQL database and use SQL queries to profile the data. This step is critical: understanding your data's actual state is the foundation of any data quality initiative.

**What you'll learn:**
- How to connect to PostgreSQL from Python
- SQL-based data profiling techniques
- Identifying quality issues: nulls, duplicates, temporal violations, referential integrity
- Why manual profiling matters before automated validation

## Environment Check

First, verify that Great Expectations 1.x is installed (we'll use it in later exercises).

In [ ]:
import great_expectations as gx
print(f"Great Expectations version: {gx.__version__}")
assert gx.__version__.startswith("1."), f"Expected GE 1.x, got {gx.__version__}"

## Connection Setup

Define connection parameters once. The PostgreSQL service is accessible at `postgresql:5432` (Docker internal network) with the credentials from `.env`.

In [ ]:
import psycopg2
import psycopg2.extras

# Connection parameters (matches docker-compose.yml defaults)
DB_CONFIG = {
    "host": "postgresql",
    "port": 5432,
    "dbname": "quality_lab",
    "user": "lab",
    "password": "lab_password"
}

conn = psycopg2.connect(**DB_CONFIG)
print("Connected to PostgreSQL successfully!")

## Exercise 1: List All Tables

Discover what tables exist in the `quality_lab` database.

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """)
    tables = cur.fetchall()

print("Tables in quality_lab database:")
for table in tables:
    print(f"  - {table[0]}")

## Exercise 2: Profile `dirty_customers`

Examine the customer table for completeness, uniqueness, and validity issues.

### 2a: Row Count and Column Overview

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("SELECT COUNT(*) as total_rows FROM dirty_customers;")
    result = cur.fetchone()
    print(f"Total rows in dirty_customers: {result['total_rows']}")

    cur.execute("""
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_name = 'dirty_customers'
        ORDER BY ordinal_position;
    """)
    columns = cur.fetchall()
    print("\nColumns:")
    for col in columns:
        print(f"  {col['column_name']:20s} {col['data_type']:20s} nullable={col['is_nullable']}")

### 2b: NULL Counts Per Column (Completeness Check)

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) - COUNT(customer_id) AS null_customer_id,
            COUNT(*) - COUNT(email) AS null_email,
            COUNT(*) - COUNT(first_name) AS null_first_name,
            COUNT(*) - COUNT(last_name) AS null_last_name,
            COUNT(*) - COUNT(phone) AS null_phone,
            COUNT(*) - COUNT(registration_date) AS null_registration_date,
            COUNT(*) - COUNT(last_login) AS null_last_login
        FROM dirty_customers;
    """)
    result = cur.fetchone()
    print("NULL counts per column:")
    for col, count in result.items():
        if col != 'total_rows':
            indicator = " <-- QUALITY ISSUE" if count > 0 else ""
            print(f"  {col:25s}: {count}{indicator}")

### 2c: Duplicate Customer IDs (Uniqueness Check)

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT customer_id, COUNT(*) AS occurrences
        FROM dirty_customers
        GROUP BY customer_id
        HAVING COUNT(*) > 1
        ORDER BY customer_id;
    """)
    duplicates = cur.fetchall()
    print(f"Duplicate customer_id values: {len(duplicates)} groups")
    for dup in duplicates:
        print(f"  customer_id={dup['customer_id']}: {dup['occurrences']} occurrences")

### 2d: Invalid Email Formats (Validity Check)

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT customer_id, email
        FROM dirty_customers
        WHERE email IS NOT NULL
          AND email !~ '^[^@\s]+@[^@\s]+\.[^@\s]+$'
        ORDER BY customer_id;
    """)
    invalid = cur.fetchall()
    print(f"Invalid email formats: {len(invalid)} rows")
    for row in invalid:
        print(f"  customer_id={row['customer_id']}: '{row['email']}'")

### 2e: Temporal Violations (Accuracy Check)

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    # Future registration dates
    cur.execute("""
        SELECT customer_id, email, registration_date
        FROM dirty_customers
        WHERE registration_date > CURRENT_TIMESTAMP
        ORDER BY customer_id;
    """)
    future_reg = cur.fetchall()
    print(f"Future registration dates: {len(future_reg)} rows")
    for row in future_reg:
        print(f"  customer_id={row['customer_id']}: registered {row['registration_date']}")

    # last_login before registration_date
    cur.execute("""
        SELECT customer_id, email, registration_date, last_login
        FROM dirty_customers
        WHERE last_login < registration_date
        ORDER BY customer_id;
    """)
    time_travel = cur.fetchall()
    print(f"\nLast login before registration: {len(time_travel)} rows")
    for row in time_travel:
        print(f"  customer_id={row['customer_id']}: registered {row['registration_date']}, last_login {row['last_login']}")

## Exercise 3: Profile `dirty_orders`

Examine the orders table for referential integrity, temporal violations, and range issues.

### 3a: Row Count and Basic Stats

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) - COUNT(order_date) AS null_order_date,
            MIN(total_amount) AS min_amount,
            MAX(total_amount) AS max_amount,
            COUNT(CASE WHEN total_amount < 0 THEN 1 END) AS negative_amounts,
            COUNT(CASE WHEN total_amount = 0 THEN 1 END) AS zero_amounts
        FROM dirty_orders;
    """)
    result = cur.fetchone()
    print("dirty_orders overview:")
    for col, val in result.items():
        print(f"  {col:25s}: {val}")

### 3b: Orphan Customer IDs (Referential Integrity Check)

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT DISTINCT o.customer_id
        FROM dirty_orders o
        LEFT JOIN dirty_customers c ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
        ORDER BY o.customer_id;
    """)
    orphans = cur.fetchall()
    print(f"Orphan customer_id values (no matching customer): {len(orphans)}")
    for row in orphans:
        print(f"  customer_id={row['customer_id']}")

### 3c: Temporal Violations (delivery before order, future dates)

In [ ]:
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    # Delivery before order
    cur.execute("""
        SELECT order_id, order_date, delivery_date
        FROM dirty_orders
        WHERE delivery_date < order_date
        ORDER BY order_id;
    """)
    temporal = cur.fetchall()
    print(f"Delivery before order date: {len(temporal)} rows")
    for row in temporal:
        print(f"  order_id={row['order_id']}: ordered {row['order_date']}, delivered {row['delivery_date']}")

    # Future order dates
    cur.execute("""
        SELECT order_id, order_date
        FROM dirty_orders
        WHERE order_date > CURRENT_TIMESTAMP
        ORDER BY order_id;
    """)
    future = cur.fetchall()
    print(f"\nFuture order dates: {len(future)} rows")
    for row in future:
        print(f"  order_id={row['order_id']}: order_date={row['order_date']}")

### 3d: Invalid Status Values and Duplicate Order IDs

In [ ]:
VALID_STATUSES = ('pending', 'processing', 'shipped', 'delivered', 'cancelled')

with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    # Invalid status values
    cur.execute("""
        SELECT order_id, status
        FROM dirty_orders
        WHERE status NOT IN ('pending', 'processing', 'shipped', 'delivered', 'cancelled')
        ORDER BY order_id;
    """)
    invalid = cur.fetchall()
    print(f"Invalid status values: {len(invalid)} rows")
    for row in invalid:
        print(f"  order_id={row['order_id']}: status='{row['status']}'")

    # Duplicate order IDs
    cur.execute("""
        SELECT order_id, COUNT(*) AS occurrences
        FROM dirty_orders
        GROUP BY order_id
        HAVING COUNT(*) > 1
        ORDER BY order_id;
    """)
    dup_orders = cur.fetchall()
    print(f"\nDuplicate order_id values: {len(dup_orders)} groups")
    for row in dup_orders:
        print(f"  order_id={row['order_id']}: {row['occurrences']} occurrences")

## Exercise 4: Quality Issue Summary

Based on your profiling above, fill in the summary of all discovered quality issues.

**Your task:** Edit the cells below to document what you found.

### dirty_customers Issues

| Quality Dimension | Issue Description | Rows Affected |
|---|---|---|
| Completeness | _[describe NULL fields found]_ | _[count]_ |
| Uniqueness | _[describe duplicates found]_ | _[count]_ |
| Validity | _[describe format violations]_ | _[count]_ |
| Accuracy | _[describe temporal/future issues]_ | _[count]_ |

### dirty_orders Issues

| Quality Dimension | Issue Description | Rows Affected |
|---|---|---|
| Referential Integrity | _[describe orphan records]_ | _[count]_ |
| Temporal Consistency | _[describe date violations]_ | _[count]_ |
| Validity | _[describe status violations]_ | _[count]_ |
| Range | _[describe amount violations]_ | _[count]_ |

## Cleanup

Close the database connection.

In [ ]:
conn.close()
print("Connection closed. Proceed to Notebook 02 for Great Expectations validation.")